In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
%pip install -q mcp arxiv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 53.0 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -q -U --force-reinstall --no-cache-dir transformers tokenizers
%pip install -q marker-pdf
%pip install -q mcp arxiv langchain-mcp-adapters langchain-groq langgraph \
    chromadb sentence-transformers rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 241.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 178.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 190.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 257.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 289.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 229.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 307.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 399.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 389.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 398.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 244.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 283.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [4]:
%%writefile /kaggle/working/arxiv_server.py
import arxiv
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("arxiv-search")         

@mcp.tool()
def search_papers(query: str, max_results: int = 5) -> list[dict]:
    """Search arXiv for papers matching the query. Returns title,
    authors, year, abstract and pdf_url for each result."""
    results = arxiv.Client().results(
        arxiv.Search(query=query, max_results=max_results)
    )
    return [{
        "title": r.title,
        "authors": [a.name for a in r.authors][:3],
        "year": r.published.year,
        "abstract": r.summary[:400],
        "pdf_url": r.pdf_url,
    } for r in results]

if __name__ == "__main__":
    mcp.run(transport="stdio")         

Writing /kaggle/working/arxiv_server.py


In [5]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(
    command="python",
    args=["/kaggle/working/arxiv_server.py"],   # how to launch the server
)

async with stdio_client(server) as (read, write):        # spawns the subprocess
    async with ClientSession(read, write) as session:
        await session.initialize()                        # MCP handshake

        tools = await session.list_tools()                # "what can you do?"
        for t in tools.tools:
            print("TOOL:", t.name, "—", t.description[:60])

        result = await session.call_tool(                 # "do this"
            "search_papers",
            {"query": "corrective retrieval augmented generation",
             "max_results": 3},
        )
        print(result.content[0].text)

TOOL: search_papers — Search arXiv for papers matching the query. Returns title,
 
{
  "title": "AR-RAG: Autoregressive Retrieval Augmentation for Image Generation",
  "authors": [
    "Jingyuan Qi",
    "Zhiyang Xu",
    "Qifan Wang"
  ],
  "year": 2025,
  "abstract": "We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, ",
  "pdf_url": "https://arxiv.org/pdf/2506.06962v3"
}


In [6]:
%pip install -q langchain-mcp-adapters langchain-groq langgraph

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

# sanity check — never print the full key
print("key loaded:", os.environ["GROQ_API_KEY"][:6] + "...")

key loaded: gsk_uD...


In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent          # new import — see note below
from langchain_groq import ChatGroq

client = MultiServerMCPClient({
    "arxiv": {"command": "python",
              "args": ["/kaggle/working/arxiv_server.py"],
              "transport": "stdio"}
})
tools = await client.get_tools()

agent = create_agent(
    ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2),
    tools,
    system_prompt=("You are a research assistant. The ONLY tool available is "
                   "search_papers(query, max_results), which searches arXiv. "
                   "Never call any other tool name. If no tool is needed, just answer.")
)

out = await agent.ainvoke({"messages":
    "Use the search_papers tool to find recent papers on self-correcting "
    "retrieval-augmented generation, then summarize the top result."})
print(out["messages"][-1].content)

for attempt in range(3):
    try:
        out = await agent.ainvoke({"messages": "..."})
        break
    except Exception as e:
        if "tool_use_failed" not in str(e) or attempt == 2:
            raise
        print(f"malformed tool call, retrying ({attempt+1})...")

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


The top result is the paper "AR-RAG: Autoregressive Retrieval Augmentation for Image Generation" by Jingyuan Qi, Zhiyang Xu, and Qifan Wang, published in 2025. This paper introduces a new paradigm called Autoregressive Retrieval Augmentation (AR-RAG) that enhances image generation by incorporating nearest neighbor retrievals at the patch level in an autoregressive manner. Unlike prior methods, AR-RAG performs context-aware retrievals at each generation step, allowing for more dynamic and adaptive generation. The paper can be found at https://arxiv.org/pdf/2506.06962v3.


In [9]:
from pathlib import Path
import shutil

STORE = Path("/kaggle/working/store")
SNAPSHOT = Path("/kaggle/input/agentic-research-store")  # snapshot dataset (later)

def init_store():
    for sub in ["chroma", "pdfs", "cache/s2"]:
        (STORE / sub).mkdir(parents=True, exist_ok=True)

def restore_store():
    if SNAPSHOT.exists():
        shutil.copytree(SNAPSHOT, STORE, dirs_exist_ok=True)
        return "restored from snapshot"
    init_store()
    return "fresh store (no snapshot attached)"

print(restore_store())
print("STORE =", STORE, "| exists:", STORE.exists())

fresh store (no snapshot attached)
STORE = /kaggle/working/store | exists: True


In [10]:
import hashlib, json, datetime
from pathlib import Path

MANIFEST_PATH = STORE / "manifest.json"
CURRENT_EMBEDDER = "BAAI/bge-base-en-v1.5"   # the guard value
MIN_PARSE_CHARS = 500                         # quarantine threshold

# ---------- load / save ----------

def load_manifest() -> dict:
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {"embedder": CURRENT_EMBEDDER, "docs": {}}

def save_manifest(m: dict):
    MANIFEST_PATH.write_text(json.dumps(m, indent=2))

# ---------- identity ----------

def compute_doc_id(pdf_path: Path) -> str:
    return hashlib.sha256(pdf_path.read_bytes()).hexdigest()[:16]

# ---------- the guard ----------

def check_embedder(m: dict):
    if m["docs"] and m["embedder"] != CURRENT_EMBEDDER:
        raise RuntimeError(
            f"Index was built with {m['embedder']} but current embedder is "
            f"{CURRENT_EMBEDDER}. Vectors are incompatible - re-index from "
            f"scratch (delete store/chroma and manifest) before continuing.")

# ---------- main entry ----------

def ingest_pdf(pdf_path: Path, parse_fn, chunk_and_index_fn=None) -> str:
    """Returns one of: 'skipped', 'quarantined', 'ingested'.
    parse_fn(pdf_path) -> markdown string  (your marker call)
    chunk_and_index_fn(markdown, record) -> n_chunks  (plugs in at Step 4)
    """
    m = load_manifest()
    check_embedder(m)

    doc_id = compute_doc_id(pdf_path)
    if doc_id in m["docs"]:
        print(f"  skip (already {m['docs'][doc_id]['status']}): {pdf_path.name}")
        return "skipped"

    markdown = parse_fn(pdf_path)

    record = {
        "doc_id": doc_id,
        "filename": pdf_path.name,
        "title": next((l.lstrip("# ").strip() for l in markdown.splitlines()
                       if l.startswith("# ")), pdf_path.stem),
        "ingested_at": datetime.datetime.utcnow().isoformat(),
        "n_chars": len(markdown),
    }

    if len(markdown) < MIN_PARSE_CHARS:
        record["status"] = "quarantined"
        print(f"  QUARANTINED ({len(markdown)} chars): {pdf_path.name}")
    else:
        if chunk_and_index_fn is not None:
            record["n_chunks"] = chunk_and_index_fn(markdown, record)
        record["status"] = "ingested"
        print(f"  ingested: {record['title'][:60]}")

    m["docs"][doc_id] = record
    save_manifest(m)
    return record["status"]

def ingest_folder(folder: Path, parse_fn, chunk_and_index_fn=None):
    results = [ingest_pdf(p, parse_fn, chunk_and_index_fn)
               for p in sorted(folder.glob("*.pdf"))]
    print(f"\n{results.count('ingested')} ingested, "
          f"{results.count('skipped')} skipped, "
          f"{results.count('quarantined')} quarantined")

def print_manifest_summary():
    m = load_manifest()
    print(f"embedder: {m['embedder']}  |  {len(m['docs'])} documents")
    for d in m["docs"].values():
        print(f"  [{d['status']:11s}] {d.get('n_chunks','-'):>4} chunks  "
              f"{d['title'][:55]}")

In [11]:
import urllib.request

papers = {
    "crag.pdf":     "https://arxiv.org/pdf/2401.15884",   # Corrective RAG
    "self_rag.pdf": "https://arxiv.org/pdf/2310.11511",   # Self-RAG
}
for name, url in papers.items():
    urllib.request.urlretrieve(url, STORE / "pdfs" / name)
    print("downloaded", name)

downloaded crag.pdf
downloaded self_rag.pdf


In [12]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

_models = create_model_dict()          # loads once; slow first time

def parse_with_marker(pdf_path: Path) -> str:
    converter = PdfConverter(artifact_dict=_models)
    rendered = converter(str(pdf_path))
    markdown, _, _ = text_from_rendered(rendered)
    return markdown

2026-06-24 10:01:53.237342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782295313.400373      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782295313.448233      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782295313.828490      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782295313.828526      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782295313.828528      58 computation_placer.cc:177] computation placer alr

In [13]:
md = parse_with_marker(STORE / "pdfs" / "crag.pdf")
print(md[:3000])

Recognizing Text: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]
Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7d1b5f6b1ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /usr/local/lib

# Corrective Retrieval Augmented Generation

Shi-Qi Yan<sup>1</sup>\*, Jia-Chen Gu<sup>2</sup>\*, Yun Zhu<sup>3</sup> , Zhen-Hua Ling<sup>1</sup>

National Engineering Research Center of Speech and Language Information Processing, University of Science and Technology of China, Hefei, China Department of Computer Science, University of California, Los Angeles Google DeepMind

yansiki@mail.ustc.edu.cn, gujc@ucla.edu, yunzhu@google.com, zhling@ustc.edu.cn

# Abstract

Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured solely by the parametric knowledge they encapsulate. Although retrieval-augmented generation (RAG) is a practicable complement to LLMs, it relies heavily on the relevance of retrieved documents, raising concerns about how the model behaves if retrieval goes wrong. To this end, we propose the Corrective Retrieval Augmented Generation (CRAG) to improve the robustness of generation. Specifically, a lightweight 

In [14]:
ingest_folder(STORE / "pdfs", parse_with_marker)   # run 1

Recognizing Text: 100%|██████████| 34/34 [00:02<00:00, 11.34it/s]


  ingested: Corrective Retrieval Augmented Generation


/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),
Recognizing Text: 100%|██████████| 62/62 [00:04<00:00, 13.23it/s]


  ingested: SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU

2 ingested, 0 skipped, 0 quarantined


/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


In [15]:
ingest_folder(STORE / "pdfs", parse_with_marker)
print_manifest_summary()

  skip (already ingested): crag.pdf
  skip (already ingested): self_rag.pdf

0 ingested, 2 skipped, 0 quarantined
embedder: BAAI/bge-base-en-v1.5  |  2 documents
  [ingested   ]    - chunks  Corrective Retrieval Augmented Generation
  [ingested   ]    - chunks  SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE 


In [55]:
import re
import hashlib
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
MAX_TOKENS, OVERLAP_TOKENS = 440, 60     # 440 + contextual header (~25) stays under BGE's 512

def ntok(s):
    return len(tok.encode(s, add_special_tokens=False))

def clean_markdown(md: str) -> str:
    md = re.sub(r"!\[.*?\]\(.*?\)", "", md)            # image artifacts
    md = re.sub(r"<span[^>]*>|</span>", "", md)         # marker's anchor spans
    md = re.sub(r"<sup>.*?</sup>", "", md)              # superscript noise
    return md

def classify_section(path: str) -> str:
    p = path.lower()
    for key in ["abstract", "introduction", "related", "method", "approach",
                "experiment", "result", "discussion", "conclusion", "reference",
                "acknowledg", "appendix"]:
        if key in p:
            return key
    return "body"

def split_by_headers(md: str):
    blocks, path, buf = [], ["PREAMBLE"], []
    for line in md.splitlines():
        m = re.match(r"^(#{1,4})\s+(.*)", line)
        if m:
            if buf:
                blocks.append((" > ".join(path), "\n".join(buf).strip()))
                buf = []
            level, title = len(m.group(1)), m.group(2).strip()
            path = path[:level-1] + [title] if level > 1 else [title]
        else:
            buf.append(line)
    if buf:
        blocks.append((" > ".join(path), "\n".join(buf).strip()))
    return [b for b in blocks if b[1]]

def hard_split(p: str) -> list[str]:
    """Split an oversized paragraph at sentence boundaries; token-slice
    anything that still won't fit (tables, equation blocks)."""
    if ntok(p) <= MAX_TOKENS:
        return [p]
    parts, cur, cur_t = [], [], 0
    for sent in re.split(r"(?<=[.!?])\s+", p):
        st = ntok(sent)
        if cur and cur_t + st > MAX_TOKENS:
            parts.append(" ".join(cur))
            cur, cur_t = [], 0
        cur.append(sent)
        cur_t += st
    if cur:
        parts.append(" ".join(cur))
    # fallback for punctuation-free blobs
    final = []
    for part in parts:
        ids = tok.encode(part, add_special_tokens=False)
        if len(ids) <= MAX_TOKENS:
            final.append(part)
        else:
            final += [tok.decode(ids[i:i + MAX_TOKENS])
                      for i in range(0, len(ids), MAX_TOKENS)]
    return final

def chunk_markdown(md: str, doc: dict) -> list[dict]:
    md = clean_markdown(md)
    out = []
    for sec_path, text in split_by_headers(md):
        stype = classify_section(sec_path)
        if stype in ("reference", "acknowledg", "appendix") or sec_path == "PREAMBLE":
            continue
        

        paras = [piece for p in text.split("\n\n") if p.strip()
                 for piece in hard_split(p)]

        chunks_here, cur, cur_t = [], [], 0
        for p in paras:
            pt = ntok(p)
            if cur and cur_t + pt > MAX_TOKENS:
                chunks_here.append("\n\n".join(cur))
                tail, t = [], 0
                for q in reversed(cur):
                    if t + ntok(q) > OVERLAP_TOKENS:
                        break
                    tail.insert(0, q)
                    t += ntok(q)
                # shed overlap if the seed itself would bust the budget
                while tail and t + pt > MAX_TOKENS:
                    t -= ntok(tail.pop(0))
                cur, cur_t = tail + [p], t + pt
            else:
                cur.append(p)
                cur_t += pt
        if cur:
            chunks_here.append("\n\n".join(cur))

        for i, c in enumerate(chunks_here):
            if ntok(c) < 30:
                continue
            out.append({
                "id": f'{doc["doc_id"]}::chunk_{len(out):04d}',
                "text": c,
                "embed_text": f'{doc["title"]} — {sec_path}\n\n{c}',
                "metadata": {"doc_id": doc["doc_id"],
                             "title": doc["title"][:80],
                             "section": sec_path[:80],
                             "section_type": stype,
                             "chunk_index": i},
            })
    return out

In [17]:
from sentence_transformers import SentenceTransformer
import chromadb

emb_model = SentenceTransformer("BAAI/bge-base-en-v1.5")   # GPU auto-detected
Q_PREFIX = "Represent this sentence for searching relevant passages: "

chroma = chromadb.PersistentClient(path=str(STORE / "chroma"))
collection = chroma.get_or_create_collection("papers", metadata={"hnsw:space": "cosine"})

def chunk_and_index(markdown: str, record: dict) -> int:
    chunks = chunk_markdown(markdown, record)
    if not chunks: return 0
    vecs = emb_model.encode([c["embed_text"] for c in chunks],
                            normalize_embeddings=True, show_progress_bar=False)
    collection.add(
        ids=[c["id"] for c in chunks],
        embeddings=vecs.tolist(),
        documents=[c["text"] for c in chunks],        # clean text for the LLM
        metadatas=[c["metadata"] for c in chunks],
    )
    return len(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
import shutil
def reset_index():
    if MANIFEST_PATH.exists(): MANIFEST_PATH.unlink()
    chroma.delete_collection("papers")
    globals()["collection"] = chroma.get_or_create_collection(
        "papers", metadata={"hnsw:space": "cosine"})
    print("index + manifest wiped")

reset_index()
ingest_folder(STORE / "pdfs", parse_with_marker, chunk_and_index)
print_manifest_summary()

index + manifest wiped


Recognizing Text: 100%|██████████| 34/34 [00:02<00:00, 11.35it/s]
/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),
Token indices sequence length is longer than the specified maximum sequence length for this model (688 > 512). Running this sequence through the model will result in indexing errors


  ingested: Corrective Retrieval Augmented Generation


Recognizing Text: 100%|██████████| 62/62 [00:04<00:00, 13.33it/s]
/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU

2 ingested, 0 skipped, 0 quarantined
embedder: BAAI/bge-base-en-v1.5  |  2 documents
  [ingested   ]   49 chunks  Corrective Retrieval Augmented Generation
  [ingested   ]   83 chunks  SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE 


In [19]:
import numpy as np
data = collection.get(include=["documents", "metadatas"])
lens = [ntok(d) for d in data["documents"]]
print(f"{len(lens)} chunks | tokens: min {min(lens)}, "
      f"median {int(np.median(lens))}, max {max(lens)}")

# expose the offenders
for d, m in zip(data["documents"], data["metadatas"]):
    if ntok(d) > MAX_TOKENS:
        print("\nOVERSIZED:", m["section"], "|", ntok(d), "tokens")
        print(d[:300])

132 chunks | tokens: min 32, median 287, max 440


In [20]:
MD_CACHE = STORE / "markdown"; MD_CACHE.mkdir(exist_ok=True)

def parse_cached(pdf_path: Path) -> str:
    cache = MD_CACHE / (compute_doc_id(pdf_path) + ".md")
    if cache.exists():
        return cache.read_text()
    md = parse_with_marker(pdf_path)
    cache.write_text(md)
    return md

In [21]:
reset_index()
ingest_folder(STORE / "pdfs", parse_cached, chunk_and_index)
print_manifest_summary()

index + manifest wiped


Recognizing Text: 100%|██████████| 34/34 [00:02<00:00, 11.34it/s]
/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: Corrective Retrieval Augmented Generation


Recognizing Text: 100%|██████████| 62/62 [00:04<00:00, 13.24it/s]
/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU

2 ingested, 0 skipped, 0 quarantined
embedder: BAAI/bge-base-en-v1.5  |  2 documents
  [ingested   ]   49 chunks  Corrective Retrieval Augmented Generation
  [ingested   ]   83 chunks  SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE 


In [22]:
data = collection.get(include=["documents", "metadatas"])
lens = [ntok(d) for d in data["documents"]]
print(f"{len(lens)} chunks | min {min(lens)}, median {int(np.median(lens))}, max {max(lens)}")
assert max(lens) <= MAX_TOKENS, "still oversized!"

132 chunks | min 32, median 287, max 440


In [23]:
def smoke(q):
    qv = emb_model.encode(Q_PREFIX + q, normalize_embeddings=True)
    res = collection.query(query_embeddings=[qv.tolist()], n_results=3)
    print("\nQ:", q)
    for doc, m in zip(res["documents"][0], res["metadatas"][0]):
        print("  →", m["title"][:40], "|", m["section"][:45])
        print("    ", doc[:160].replace("\n", " "))

smoke("What corrective actions does CRAG take when retrieval quality is low?")
smoke("How does Self-RAG use reflection tokens to critique its own generation?")


Q: What corrective actions does CRAG take when retrieval quality is low?
  → Corrective Retrieval Augmented Generatio | 1 Introduction
     On account of the above issues, this paper particularly studies the scenarios where the retriever returns inaccurate results. A method named Corrective Retrieva
  → Corrective Retrieval Augmented Generatio | 6 Conclusion & Limitation
     This paper studies the problem where RAG-based approaches are challenged if retrieval goes wrong, thereby exposing inaccurate and misleading knowledge to genera
  → Corrective Retrieval Augmented Generatio | 5 Experiments
     We conducted experiments to extensively demonstrate CRAG's adaptability to RAG-based approaches and its generalizability across both shortand long-form generati

Q: How does Self-RAG use reflection tokens to critique its own generation?
  → SELF-RAG: LEARNING TO RETRIEVE, GENERATE | A SELF-RAG DETAILS > A.1 REFLECTION TOKENS.
     Definitions of reflection tokens. Below, we provide a detail

In [24]:
def show_chunks(title_substr):
    data = collection.get(include=["documents", "metadatas"])
    for cid, d, m in zip(data["ids"], data["documents"], data["metadatas"]):
        if title_substr.lower() in m["title"].lower():
            print(f"{cid}\n  [{m['section_type']}] {m['section'][:60]}\n  {d[:120]}...\n")

show_chunks("Corrective")   # then show_chunks("SELF-RAG")

975aa1fd3c1b6031::eb581f51::0
  [body] Corrective Retrieval Augmented Generation
  Shi-Qi Yan\*, Jia-Chen Gu\*, Yun Zhu , Zhen-Hua Ling

National Engineering Research Center of Speech and Language Inform...

975aa1fd3c1b6031::e353dbe4::0
  [abstract] Abstract
  Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured s...

975aa1fd3c1b6031::8d2142fa::0
  [introduction] 1 Introduction
  Large language models (LLMs) have attracted increasing attention and exhibited impressive abilities to understand instru...

975aa1fd3c1b6031::8d2142fa::1
  [introduction] 1 Introduction
  **Retrieved Documents** Prior research has introduced the retrieval techniques to incorporate the knowledge relevant to ...

975aa1fd3c1b6031::8d2142fa::2
  [introduction] 1 Introduction
   Equal contribution.

The code is available at [github.com/HuskyInSalt/CRAG](https://github.com/HuskyInSalt/CRAG)

a sub...

975aa1fd3c1b6031::8d2142fa::3
  [introduction]

In [25]:
show_chunks("SELF-RAG")

d9eaa1398abac0df::dfd980ba::0
  [body] SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU
  Akari Asai† , Zeqiu Wu† , Yizhong Wang†§, Avirup Sil‡ , Hannaneh Hajishirzi†§ †University of Washington §Allen Institute...

d9eaa1398abac0df::f387fb12::0
  [abstract] ABSTRACT
  Despite their remarkable capabilities, large language models (LLMs) often produce responses containing factual inaccurac...

d9eaa1398abac0df::310aec30::0
  [introduction] 1 INTRODUCTION
  State-of-the-art LLMs continue to struggle with factual errors [\(Mallen et al., 2023;](#page-12-0) [Min et al., 2023\)]...

d9eaa1398abac0df::310aec30::1
  [introduction] 1 INTRODUCTION
  Reflection tokens are categorized into *retrieval* and *critique* tokens to indicate the need for retrieval and its gene...

d9eaa1398abac0df::310aec30::2
  [introduction] 1 INTRODUCTION
  SELF-RAG trains an arbitrary LM to generate text with reflection tokens by unifying them as the next token prediction fr...

d9eaa1398abac0df::310aec30:

In [26]:
C = "975aa1fd3c1b6031"   # CRAG
S = "d9eaa1398abac0df"   # Self-RAG

EVAL_SET = [
  # ---------- factoid ----------
  {"q": "What three retrieval quality judgments does CRAG's evaluator produce?",
   "relevant_ids": [f"{C}::c9d67925::0", f"{C}::c9d67925::1"], "answerable": True},
  {"q": "What corrective action does CRAG trigger when retrieval is judged Incorrect?",
   "relevant_ids": [f"{C}::c9d67925::1", f"{C}::aff5dedc::0"], "answerable": True},
  {"q": "What is the decompose-then-recompose algorithm in CRAG used for?",
   "relevant_ids": [f"{C}::98392ba7::0"], "answerable": True},
  {"q": "What model is CRAG's retrieval evaluator fine-tuned from?",
   "relevant_ids": [f"{C}::8fabe20f::0", f"{C}::0f322ed5::0"], "answerable": True},
  {"q": "What are the four types of reflection tokens in Self-RAG?",
   "relevant_ids": [f"{S}::1f63c8bd::2", f"{S}::e4c534c1::0", f"{S}::e4c534c1::1"],
   "answerable": True},
  {"q": "How is the critic model in Self-RAG trained and where does its training data come from?",
   "relevant_ids": [f"{S}::906aa159::0", f"{S}::906aa159::2"], "answerable": True},
  # ---------- synthesis ----------
  {"q": "What role does web search play in CRAG's pipeline?",
   "relevant_ids": [f"{C}::aff5dedc::0", f"{C}::0198e4dc::0"], "answerable": True},
  {"q": "How does Self-RAG use reflection tokens to control generation at inference time?",
   "relevant_ids": [f"{S}::93ac9dd8::1", f"{S}::30c1c079::0", f"{S}::310aec30::1"],
   "answerable": True},
  {"q": "How do CRAG and Self-RAG differ in how they judge whether retrieval was good?",
   "relevant_ids": [f"{C}::0f322ed5::0", f"{S}::310aec30::1", f"{S}::93ac9dd8::1"],
   "answerable": True},
  # ---------- unanswerable ----------
  {"q": "What does the RAPTOR paper propose for hierarchical summarization?",
   "relevant_ids": [], "answerable": False},
  {"q": "How does GraphRAG build community summaries over a document corpus?",
   "relevant_ids": [], "answerable": False},
  {"q": "What dollar cost did the CRAG authors report for their GPT-4 API usage?",
   "relevant_ids": [], "answerable": False},
]
print(len(EVAL_SET), "questions,",
      sum(e["answerable"] for e in EVAL_SET), "answerable")

12 questions, 9 answerable


In [27]:
def dense_retrieve(q, top_k=5):
    qv = emb_model.encode(Q_PREFIX + q, normalize_embeddings=True)
    res = collection.query(query_embeddings=[qv.tolist()], n_results=top_k)
    return res["ids"][0]

def evaluate(retrieve_fn, k=5, label=""):
    hits, mrrs = [], []
    for ex in EVAL_SET:
        if not ex["answerable"]:
            continue
        ids = retrieve_fn(ex["q"], top_k=k)
        rel = set(ex["relevant_ids"])
        hits.append(float(any(i in rel for i in ids)))
        rank = next((j + 1 for j, i in enumerate(ids) if i in rel), None)
        mrrs.append(1 / rank if rank else 0.0)
    print(f"{label:30s} hit@{k}: {np.mean(hits):.3f}   MRR: {np.mean(mrrs):.3f}")
    return np.mean(hits), np.mean(mrrs)

In [28]:
evaluate(dense_retrieve, k=5, label="dense (BGE + struct chunks)")

dense (BGE + struct chunks)    hit@5: 0.889   MRR: 0.587


(np.float64(0.8888888888888888), np.float64(0.587037037037037))

In [29]:
def show_misses(retrieve_fn, k=5):
    for ex in EVAL_SET:
        if not ex["answerable"]:
            continue
        ids = retrieve_fn(ex["q"], top_k=k)
        rel = set(ex["relevant_ids"])
        rank = next((j+1 for j, i in enumerate(ids) if i in rel), None)
        print(f"rank {rank if rank else 'MISS':>4} | {ex['q'][:70]}")

show_misses(dense_retrieve)

rank    5 | What three retrieval quality judgments does CRAG's evaluator produce?
rank    1 | What corrective action does CRAG trigger when retrieval is judged Inco
rank MISS | What is the decompose-then-recompose algorithm in CRAG used for?
rank    1 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank    2 | How is the critic model in Self-RAG trained and where does its trainin
rank    1 | What role does web search play in CRAG's pipeline?
rank    3 | How does Self-RAG use reflection tokens to control generation at infer
rank    4 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [30]:
import re
from rank_bm25 import BM25Okapi

# pull corpus from Chroma (needs `collection` from Cell B above)
data = collection.get(include=["documents"])
chunk_ids, docs = data["ids"], data["documents"]
doc_lookup = dict(zip(chunk_ids, docs))

def bm25_tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

bm25 = BM25Okapi([bm25_tok(d) for d in docs])

def hybrid_retrieve(q, top_k=5, k_each=20):
    qv = emb_model.encode(Q_PREFIX + q, normalize_embeddings=True)
    dense = collection.query(query_embeddings=[qv.tolist()], n_results=k_each)["ids"][0]
    scores = bm25.get_scores(bm25_tok(q))
    sparse = [chunk_ids[i] for i in
              sorted(range(len(scores)), key=lambda i: -scores[i])[:k_each]]
    K, fused = 60, {}
    for lst in (dense, sparse):
        for r, cid in enumerate(lst):
            fused[cid] = fused.get(cid, 0) + 1/(K + r + 1)
    return sorted(fused, key=fused.get, reverse=True)[:top_k]

In [31]:
evaluate(hybrid_retrieve, k=5, label="hybrid (fixed tokenizer)")
show_misses(hybrid_retrieve)

hybrid (fixed tokenizer)       hit@5: 0.778   MRR: 0.722
rank MISS | What three retrieval quality judgments does CRAG's evaluator produce?
rank    2 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    1 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    1 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank    1 | How is the critic model in Self-RAG trained and where does its trainin
rank    1 | What role does web search play in CRAG's pipeline?
rank    1 | How does Self-RAG use reflection tokens to control generation at infer
rank MISS | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [32]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")

def rerank_retrieve(q, top_k=5):
    cands = hybrid_retrieve(q, top_k=20)
    scores = reranker.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]

def rerank_dense(q, top_k=5):
    cands = dense_retrieve(q, top_k=20)
    scores = reranker.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]
for name in ["dense_retrieve", "hybrid_retrieve", "rerank_retrieve",
             "rerank_dense", "evaluate", "show_misses", "EVAL_SET"]:
    assert name in globals(), f"MISSING: {name}"
print("spine complete ✓")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

spine complete ✓


In [33]:
evaluate(rerank_retrieve, k=5, label="hybrid + cross-encoder rerank")
show_misses(rerank_retrieve)

hybrid + cross-encoder rerank  hit@5: 0.889   MRR: 0.491
rank    2 | What three retrieval quality judgments does CRAG's evaluator produce?
rank    2 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    4 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    3 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank MISS | How is the critic model in Self-RAG trained and where does its trainin
rank    2 | What role does web search play in CRAG's pipeline?
rank    1 | How does Self-RAG use reflection tokens to control generation at infer
rank    3 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [34]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-base")

def rerank_dense(q, top_k=5):
    cands = dense_retrieve(q, top_k=20)
    scores = reranker.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]

evaluate(rerank_dense, k=5, label="dense + rerank (no BM25)")
show_misses(rerank_dense)

dense + rerank (no BM25)       hit@5: 0.889   MRR: 0.593
rank    2 | What three retrieval quality judgments does CRAG's evaluator produce?
rank    2 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    2 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    3 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank MISS | How is the critic model in Self-RAG trained and where does its trainin
rank    2 | What role does web search play in CRAG's pipeline?
rank    1 | How does Self-RAG use reflection tokens to control generation at infer
rank    1 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [35]:
def inspect_top(retrieve_fn, q, k=3):
    print("Q:", q)
    for cid in retrieve_fn(q, top_k=k):
        r = collection.get(ids=[cid], include=["documents","metadatas"])
        print("  •", r["metadatas"][0]["section"][:55], "\n   ", 
              r["documents"][0][:180].replace("\n"," "), "\n")

inspect_top(rerank_retrieve, EVAL_SET[0]["q"])   # three judgments
inspect_top(rerank_retrieve, EVAL_SET[3]["q"])   # T5-large
inspect_top(rerank_retrieve, EVAL_SET[5]["q"])   # critic training (the MISS)

Q: What three retrieval quality judgments does CRAG's evaluator produce?
  • A Task Prompts > B Experiments > B.1 Tasks, Datasets an 
    CRAG was evaluated on four datasets, which are in public domain and licensed for research purposes, including:  PopQA [\(Mallen et al.,](#page-10-2) [2023\)](#page-10-2) is a *shor 

  • 3 Task Formulation > 4.2 Retrieval Evaluator > 4.3 Acti 
    Ambiguous Except for the above two situations, the remaining will be assigned to an intermediate action of Ambiguous. This generally occurs when the accuracy of the retrieval is ha 

  • Abstract 
    Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured solely by the parametric knowledge they encapsulate. Although 

Q: What model is CRAG's retrieval evaluator fine-tuned from?
  • Abstract 
    Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured solely by the parametric knowled

In [36]:
reranker2 = CrossEncoder("BAAI/bge-reranker-v2-m3")   # ~2GB download

def rerank2_retrieve(q, top_k=5):
    cands = hybrid_retrieve(q, top_k=20)
    scores = reranker2.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]

evaluate(rerank2_retrieve, k=5, label="hybrid + bge-reranker-v2-m3")
show_misses(rerank2_retrieve)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

hybrid + bge-reranker-v2-m3    hit@5: 0.778   MRR: 0.417
rank MISS | What three retrieval quality judgments does CRAG's evaluator produce?
rank    4 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    3 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    1 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank MISS | How is the critic model in Self-RAG trained and where does its trainin
rank    3 | What role does web search play in CRAG's pipeline?
rank    3 | How does Self-RAG use reflection tokens to control generation at infer
rank    2 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [37]:
evaluate(hybrid_retrieve, k=5, label="sanity: which hybrid is live?")

sanity: which hybrid is live?  hit@5: 0.778   MRR: 0.722


(np.float64(0.7777777777777778), np.float64(0.7222222222222222))

In [38]:
%pip install -q langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [39]:
from langchain_groq import ChatGroq
import json, re as _re

fast = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
big  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

GRADE_PROMPT = """Question: {question}

Retrieved passages:
{chunks}

Judge the SET of passages as a whole. Passages being on-topic is NOT enough --
they must actually contain the information needed to answer THIS question.
- "sufficient": the passages contain the information needed to answer fully
- "insufficient": on-topic but missing key parts of the answer
- "irrelevant": mostly off-topic for this question
Return ONLY JSON: {{"grade": "...", "missing": "what is missing, if anything"}}"""

def parse_json_loose(text):
    m = _re.search(r"\{.*\}", text, _re.DOTALL)
    return json.loads(m.group()) if m else {"grade": "PARSE_FAIL", "missing": text[:120]}

def grade(question, model=fast, show_chunks=False):
    ids = rerank_retrieve(question, top_k=5)
    chunks = "\n\n".join(f"[{i+1}] {doc_lookup[c][:600]}" for i, c in enumerate(ids))
    out = model.invoke(GRADE_PROMPT.format(question=question, chunks=chunks))
    verdict = parse_json_loose(out.content)
    print(f"Q: {question}\n  → grade: {verdict['grade']}\n  → missing: {verdict['missing']}\n")
    if show_chunks:
        for i, c in enumerate(ids):
            print(f"  [{i+1}]", doc_lookup[c][:100].replace("\n", " "))
    return verdict

In [40]:
print("=== 70B on the failures ===\n")
grade("What does the RAPTOR paper propose for hierarchical summarization?",
      model=big, show_chunks=True)
grade("How does GraphRAG build community summaries over a document corpus?",
      model=big, show_chunks=True)

=== 70B on the failures ===

Q: What does the RAPTOR paper propose for hierarchical summarization?
  → grade: irrelevant
  → missing: The RAPTOR paper and its proposal for hierarchical summarization are not mentioned in the provided passages.

  [1] Despite their remarkable capabilities, large language models (LLMs) often produce responses containi
  [2] Concurrent RAG work. A few concurrent works[2](#page-2-0) on RAG propose new training or prompting s
  [3] Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts
  [4] Word embedding is the collective name for a set of language modeling and feature learning techniques
  [5] This work introduces Self-Rag, a new framework to enhance the quality and factuality of LLMs through
Q: How does GraphRAG build community summaries over a document corpus?
  → grade: irrelevant
  → missing: Information about how GraphRAG builds community summaries over a document corpus

  [1] Retrieval setup details. By

{'grade': 'irrelevant',
 'missing': 'Information about how GraphRAG builds community summaries over a document corpus'}

In [41]:
print(len(EVAL_SET), "eval questions |", collection.count(), "chunks in index")

12 eval questions | 132 chunks in index


In [50]:
from langchain_groq import ChatGroq
import json, re as _re

big = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

ORCHESTRATOR_PROMPT = """You are a research strategist for a technical AI competition team.
Given a problem statement (PS), decompose it into 3-5 specific research hypotheses.

CRITICAL: Your search queries will be used on arXiv and Semantic Scholar.
The PS is about an AI/ML system. Your queries MUST use the specific technical
vocabulary of the relevant subfields. Examples of GOOD specific queries:
- "retrieval augmented generation survey"
- "corrective RAG self-evaluation retrieval"
- "LLM agent tool use reasoning"
- "GraphRAG knowledge graph community detection"
- "multi-agent LLM orchestration LangGraph"
- "scientific document parsing PDF extraction"
- "cross-encoder reranking passage retrieval"

Examples of BAD generic queries that return irrelevant papers:
- "natural language processing techniques"
- "knowledge graph construction"
- "automated literature review"
- "graph neural networks transformers"

For each hypothesis provide:
- hypothesis: a clear research question
- search_queries: 2-3 queries using arxiv-style technical terms (think: what would
  appear in the TITLE of a relevant paper?)
- section_type: "sota" / "method" / "feasibility" / "dataset"

PS:
{ps}

Return ONLY JSON: {{"hypotheses": [
  {{"hypothesis": "...", "search_queries": ["...", "..."], "section_type": "..."}},
  ...
]}}"""

def orchestrate(ps: str) -> list[dict]:
    out = big.invoke(ORCHESTRATOR_PROMPT.format(ps=ps))
    m = _re.search(r"\{.*\}", out.content, _re.DOTALL)
    parsed = json.loads(m.group())
    return parsed["hypotheses"]

TEST_PS = """Build an agentic framework for automated research synthesis in technical
competitions. The system should browse research papers on the web based on a problem
statement, construct a knowledge graph of methods and their relationships, and produce
a grounded literature review that reduces the review phase from days to minutes."""

hypotheses = orchestrate(TEST_PS)
for i, h in enumerate(hypotheses):
    print(f"\nH{i+1} [{h['section_type']}]: {h['hypothesis']}")
    for q in h["search_queries"]:
        print(f"    → {q}")


H1 [method]: Can a retrieval-augmented generation approach be used to construct a knowledge graph of methods and their relationships from a large corpus of research papers?
    → retrieval augmented generation for knowledge graph construction
    → RAG-based knowledge graph embedding
    → graph-based retrieval for literature review

H2 [sota]: How can a cross-encoder based reranking approach be used to efficiently filter and rank relevant research papers for a given problem statement?
    → cross-encoder reranking for passage retrieval
    → efficient passage retrieval with cross-encoders
    → reranking for literature review with cross-encoders

H3 [feasibility]: Is it feasible to use a multi-agent framework with large language models to automate the literature review process and reduce the review phase from days to minutes?
    → multi-agent LLM for literature review
    → LLM-based literature review automation
    → large language model orchestration for research synthesis

H4 [me

In [46]:
import arxiv
import requests
import time
import  json
from pathlib import Path

S2_BASE = "https://api.semanticscholar.org/graph/v1"
S2_CACHE = STORE / "cache" / "s2"; S2_CACHE.mkdir(parents=True, exist_ok=True)
S2_FIELDS = "title,abstract,year,externalIds,url,referenceCount,citationCount"

# ---------- rate limiter ----------
_last_call = 0
def rate_limit(delay=1.0):
    global _last_call
    elapsed = time.time() - _last_call
    if elapsed < delay:
        time.sleep(delay - elapsed)
    _last_call = time.time()

# ---------- Semantic Scholar with disk cache ----------
def s2_get(endpoint, params=None):
    cache_key = endpoint.replace("/", "_") + "_" + json.dumps(params or {}, sort_keys=True)
    cache_key = "".join(c for c in cache_key if c.isalnum() or c in "_-")[:150]
    cache_path = S2_CACHE / f"{cache_key}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    rate_limit()
    try:
        r = requests.get(f"{S2_BASE}{endpoint}", params=params, timeout=15)
        if r.status_code == 429:
            print("    S2 rate limited, waiting 5s...")
            time.sleep(5); r = requests.get(f"{S2_BASE}{endpoint}", params=params, timeout=15)
        if r.status_code != 200:
            return None
        data = r.json()
        cache_path.write_text(json.dumps(data))
        return data
    except Exception as e:
        print(f"    S2 error: {e}")
        return None

def s2_search(query, limit=5):
    data = s2_get("/paper/search", {"query": query, "limit": limit, "fields": S2_FIELDS})
    return data.get("data", []) if data else []

def s2_references(paper_id, limit=20):
    data = s2_get(f"/paper/{paper_id}/references", {"fields": S2_FIELDS, "limit": limit})
    return [r["citedPaper"] for r in (data.get("data") or [] if data else [])
            if r.get("citedPaper", {}).get("title")]

def s2_citations(paper_id, limit=20):
    data = s2_get(f"/paper/{paper_id}/citations", {"fields": S2_FIELDS, "limit": limit})
    return [r["citingPaper"] for r in (data.get("data") or [] if data else [])
            if r.get("citingPaper", {}).get("title")]

# ---------- arXiv search ----------
def arxiv_search(query, max_results=3):
    results = []
    try:
        for r in arxiv.Client().results(
            arxiv.Search(query=query, max_results=max_results,
                         sort_by=arxiv.SortCriterion.Relevance)):
            results.append({
                "title": r.title,
                "abstract": r.summary[:500],
                "year": r.published.year,
                "pdf_url": r.pdf_url,
                "arxiv_id": r.entry_id.split("/")[-1],
                "source": "arxiv",
            })
        time.sleep(1.0)
    except Exception as e:
        print(f"    arXiv error: {e}")
    return results

# ---------- normalize S2 paper to common format ----------
def normalize_s2(paper, source_tag="s2_search"):
    if not paper or not paper.get("title"):
        return None
    ext = paper.get("externalIds", {}) or {}
    return {
        "title": paper["title"],
        "abstract": (paper.get("abstract") or "")[:500],
        "year": paper.get("year"),
        "pdf_url": f"https://arxiv.org/pdf/{ext['ArXiv']}" if ext.get("ArXiv") else None,
        "arxiv_id": ext.get("ArXiv"),
        "s2_id": paper.get("paperId"),
        "source": source_tag,
    }

# ---------- the Librarian ----------
def librarian(hypotheses, depth=1, seeds_per_query=3, chase_top_n=3, max_total=80):
    """
    Phase 1: keyword search across arXiv + Semantic Scholar per hypothesis
    Phase 2: citation chase (refs + citers) on the top papers, pruned by depth
    Returns deduplicated paper list.
    """
    seen_titles = set()
    all_papers = []
    
    def dedup_add(paper, tag):
        if not paper or not paper.get("title"):
            return False
        key = paper["title"].lower().strip()[:80]
        if key in seen_titles or len(all_papers) >= max_total:
            return False
        seen_titles.add(key)
        paper["source_tag"] = tag
        all_papers.append(paper)
        return True
    
    # --- Phase 1: keyword search ---
    print("=== Phase 1: Keyword Search ===\n")
    for h in hypotheses:
        print(f"[{h['section_type']}] {h['hypothesis'][:65]}")
        for query in h["search_queries"]:
            # arXiv
            for p in arxiv_search(query, max_results=seeds_per_query):
                if dedup_add(p, f"arxiv|{query[:30]}"):
                    print(f"  + [{p['year']}] {p['title'][:65]}")
            # Semantic Scholar
            for sp in s2_search(query, limit=seeds_per_query):
                np = normalize_s2(sp, f"s2|{query[:30]}")
                if np and dedup_add(np, f"s2|{query[:30]}"):
                    print(f"  + [{np.get('year','')}] {np['title'][:65]}")
    
    phase1_count = len(all_papers)
    print(f"\n--- Phase 1 complete: {phase1_count} papers ---")
    
    if depth == 0:
        return all_papers
    
    # --- Phase 2: citation chase on top papers ---
    print(f"\n=== Phase 2: Citation Chase (depth={depth}) ===\n")
    chase_candidates = [p for p in all_papers if p.get("s2_id")][:chase_top_n * 2]
    if not chase_candidates:
        # try to find S2 IDs for arXiv papers
        for p in all_papers[:chase_top_n * 2]:
            if p.get("arxiv_id") and not p.get("s2_id"):
                data = s2_get(f"/paper/ArXiv:{p['arxiv_id']}", {"fields": "paperId"})
                if data:
                    p["s2_id"] = data.get("paperId")
                    chase_candidates.append(p)
    
    chased = 0
    for p in chase_candidates:
        if chased >= chase_top_n or len(all_papers) >= max_total:
            break
        if not p.get("s2_id"):
            continue
        print(f"  Chasing refs+citers of: {p['title'][:55]}")
        
        # references (backward — foundations)
        for ref in s2_references(p["s2_id"], limit=10):
            nr = normalize_s2(ref, f"ref_of|{p['title'][:20]}")
            if nr and dedup_add(nr, nr["source"]):
                print(f"    ← [{nr.get('year','')}] {nr['title'][:55]}")
        
        # citations (forward — state of the art)
        for cit in s2_citations(p["s2_id"], limit=10):
            nc = normalize_s2(cit, f"citer_of|{p['title'][:20]}")
            if nc and dedup_add(nc, nc["source"]):
                print(f"    → [{nc.get('year','')}] {nc['title'][:55]}")
        
        chased += 1
    
    print(f"\n=== Librarian done: {phase1_count} from search + "
          f"{len(all_papers) - phase1_count} from citation chase = "
          f"{len(all_papers)} total ===")
    return all_papers

# --- RUN IT ---
papers = librarian(hypotheses, depth=1, seeds_per_query=3, chase_top_n=3)

=== Phase 1: Keyword Search ===

[sota] Can a retrieval-augmented generation approach be used to construc
  + [2025] AR-RAG: Autoregressive Retrieval Augmentation for Image Generatio
  + [2025] AI Agent-Driven Framework for Automated Product Knowledge Graph C
  + [2025] Intelligent Interaction Strategies for Context-Aware Cognitive Au
    S2 rate limited, waiting 5s...
  + [2003] This paper has been withdrawn
  + [2014] Unsupervised Visual and Textual Information Fusion in Multimedia 
  + [2026] Tensor Manifold-Based Graph-Vector Fusion for AI-Native Academic 
    S2 rate limited, waiting 5s...
  + [2025] Improving Graph Embeddings in Machine Learning Using Knowledge Co
  + [2022] Knowledge Graph Curation: A Practical Framework
  + [2025] Bi-View Embedding Fusion: A Hybrid Learning Approach for Knowledg
[method] How can agentic language models be used to produce a grounded lit
  + [2025] AutoTool: Dynamic Tool Selection and Integration for Agentic Reas
  + [2026] Agentic Reasoning for 

In [51]:
import arxiv
import requests
import time
import json as _json
from pathlib import Path
from sentence_transformers import CrossEncoder

# reuse if already loaded
if "reranker" not in dir():
    reranker = CrossEncoder("BAAI/bge-reranker-base")

S2_BASE = "https://api.semanticscholar.org/graph/v1"
S2_CACHE = STORE / "cache" / "s2"; S2_CACHE.mkdir(parents=True, exist_ok=True)
S2_FIELDS = "title,abstract,year,externalIds,url,referenceCount,citationCount"

# clear stale cache entries from previous failed runs
for _f in S2_CACHE.glob("*.json"):
    if _f.read_text().strip() in ("", "None"):
        _f.unlink()

# ---------- rate limiter ----------
_last_call = 0
def rate_limit(delay=1.0):
    global _last_call
    elapsed = time.time() - _last_call
    if elapsed < delay:
        time.sleep(delay - elapsed)
    _last_call = time.time()

# ---------- Semantic Scholar with disk cache + negative caching ----------
def s2_get(endpoint, params=None):
    cache_key = endpoint.replace("/", "_") + "_" + _json.dumps(params or {}, sort_keys=True)
    cache_key = "".join(c for c in cache_key if c.isalnum() or c in "_-")[:150]
    cache_path = S2_CACHE / f"{cache_key}.json"
    if cache_path.exists():
        raw = cache_path.read_text()
        if raw.strip() == "null":
            return None                 # cached failure — don't retry
        return _json.loads(raw)
    rate_limit()
    try:
        r = requests.get(f"{S2_BASE}{endpoint}", params=params, timeout=15)
        if r.status_code == 429:
            print("    S2 rate limited, waiting 5s...")
            time.sleep(5)
            r = requests.get(f"{S2_BASE}{endpoint}", params=params, timeout=15)
        if r.status_code != 200:
            cache_path.write_text("null")       # cache the failure
            return None
        data = r.json()
        cache_path.write_text(_json.dumps(data))
        return data
    except Exception as e:
        print(f"    S2 error: {e}")
        cache_path.write_text("null")           # cache the failure
        return None

def s2_search(query, limit=5):
    data = s2_get("/paper/search", {"query": query, "limit": limit, "fields": S2_FIELDS})
    return data.get("data") or [] if data else []

def s2_references(paper_id, limit=20):
    data = s2_get(f"/paper/{paper_id}/references", {"fields": S2_FIELDS, "limit": limit})
    return [r["citedPaper"] for r in (data.get("data") or [] if data else [])
            if r.get("citedPaper", {}).get("title")]

def s2_citations(paper_id, limit=20):
    data = s2_get(f"/paper/{paper_id}/citations", {"fields": S2_FIELDS, "limit": limit})
    return [r["citingPaper"] for r in (data.get("data") or [] if data else [])
            if r.get("citingPaper", {}).get("title")]

# ---------- arXiv search ----------
def arxiv_search(query, max_results=3):
    results = []
    try:
        for r in arxiv.Client().results(
            arxiv.Search(query=query, max_results=max_results,
                         sort_by=arxiv.SortCriterion.Relevance)):
            results.append({
                "title": r.title,
                "abstract": r.summary[:500],
                "year": r.published.year,
                "pdf_url": r.pdf_url,
                "arxiv_id": r.entry_id.split("/")[-1],
                "source": "arxiv",
            })
        time.sleep(1.0)
    except Exception as e:
        print(f"    arXiv error: {e}")
    return results

# ---------- normalize S2 paper ----------
def normalize_s2(paper, source_tag="s2_search"):
    if not paper or not paper.get("title"):
        return None
    ext = paper.get("externalIds", {}) or {}
    return {
        "title": paper["title"],
        "abstract": (paper.get("abstract") or "")[:500],
        "year": paper.get("year"),
        "pdf_url": f"https://arxiv.org/pdf/{ext['ArXiv']}" if ext.get("ArXiv") else None,
        "arxiv_id": ext.get("ArXiv"),
        "s2_id": paper.get("paperId"),
        "source": source_tag,
    }

# ============================================================
# CRITIC
# ============================================================

CRITIC_PROMPT = """You are a strict relevance judge for a research synthesis system.

The team is working on this problem:
{ps}

A paper was found with this title and abstract:
TITLE: {title}
ABSTRACT: {abstract}

Question: Is this paper DIRECTLY relevant to solving the problem statement above?
A paper is relevant ONLY if it describes methods, systems, evaluations, or datasets
that the team could actually USE or BUILD UPON for their specific problem.
Papers about unrelated domains that happen to share terminology are NOT relevant.

Return ONLY JSON: {{"relevant": true/false, "reason": "one sentence why"}}"""

def critic_score(hypothesis, paper, ps=None):
    title = paper.get("title", "")
    abstract = paper.get("abstract", "")
    ce_input = f"{title}. {abstract[:400]}"
    ce_score = float(reranker.predict([(hypothesis, ce_input)])[0])

    llm_relevant = None
    llm_reason = ""

    if -2.0 < ce_score < 2.0:
        try:
            out = big.invoke(CRITIC_PROMPT.format(
                ps=ps or hypothesis, title=title, abstract=abstract[:500]))
            m = _re.search(r"\{.*\}", out.content, _re.DOTALL)
            if m:
                parsed = _json.loads(m.group())
                llm_relevant = parsed.get("relevant", False)
                llm_reason = parsed.get("reason", "")
        except Exception as e:
            llm_reason = f"LLM error: {e}"

    if ce_score >= 2.0:
        keep = True
    elif ce_score <= -2.0:
        keep = False
    elif llm_relevant is not None:
        keep = llm_relevant
    else:
        keep = ce_score > 0

    return {"ce_score": round(ce_score, 3), "llm_relevant": llm_relevant,
            "llm_reason": llm_reason, "keep": keep}

def critic_filter(papers, hypotheses, ps, verbose=True):
    hyp_map = {h["hypothesis"][:80]: h["hypothesis"] for h in hypotheses}
    kept, dropped = [], []

    for p in papers:
        source_h = p.get("source_hypothesis", "")
        if source_h in hyp_map:
            hyp = hyp_map[source_h]
        else:
            scores = [float(reranker.predict([(h["hypothesis"],
                      f"{p['title']}. {p.get('abstract','')[:300]}")])[0])
                      for h in hypotheses]
            hyp = hypotheses[scores.index(max(scores))]["hypothesis"]

        result = critic_score(hyp, p, ps=ps)
        p["critic"] = result

        if result["keep"]:
            kept.append(p)
            if verbose:
                tag = "CE" if result["llm_relevant"] is None else "LLM"
                print(f"  ✓ [{tag} {result['ce_score']:+.2f}] {p['title'][:65]}")
        else:
            dropped.append(p)
            if verbose:
                print(f"  ✗ [{result['ce_score']:+.2f}] {p['title'][:55]} "
                      f"| {result.get('llm_reason','')[:40]}")

    print(f"\n  Critic: {len(kept)} kept, {len(dropped)} dropped "
          f"({len(dropped)/(len(papers) or 1)*100:.0f}% filtered)")
    return kept

# ============================================================
# LIBRARIAN v2
# ============================================================

def librarian_v2(hypotheses, ps, depth=1, seeds_per_query=3,
                 chase_top_n=5, max_total=60):

    seen_titles = set()
    all_papers = []

    def dedup_add(paper, tag):
        if not paper or not paper.get("title"):
            return False
        key = paper["title"].lower().strip()[:80]
        if key in seen_titles or len(all_papers) >= max_total:
            return False
        seen_titles.add(key)
        paper["source_tag"] = tag
        all_papers.append(paper)
        return True

    # === Phase 1: Keyword Search ===
    print("=== Phase 1: Keyword Search ===\n")
    phase1 = []
    for h in hypotheses:
        print(f"[{h['section_type']}] {h['hypothesis'][:65]}")
        for query in h["search_queries"]:
            for p in arxiv_search(query, max_results=seeds_per_query):
                p["source_hypothesis"] = h["hypothesis"][:80]
                if dedup_add(p, f"arxiv|{query[:30]}"):
                    phase1.append(p)
                    print(f"  + [{p['year']}] {p['title'][:65]}")
            for sp in s2_search(query, limit=seeds_per_query):
                np = normalize_s2(sp, f"s2|{query[:30]}")
                if np:
                    np["source_hypothesis"] = h["hypothesis"][:80]
                    if dedup_add(np, f"s2|{query[:30]}"):
                        phase1.append(np)
                        print(f"  + [{np.get('year','')}] {np['title'][:65]}")

    print(f"\n--- Phase 1: {len(phase1)} papers found ---")

    # === Phase 2: Critic filters Phase 1 BEFORE citation chasing ===
    print(f"\n=== Phase 2: Critic Filtering Phase 1 ===\n")
    survivors = critic_filter(phase1, hypotheses, ps)

    if depth == 0:
        return survivors

    # === Phase 3: Citation Chase ONLY on survivors ===
    print(f"\n=== Phase 3: Citation Chase (depth={depth}) on "
          f"{len(survivors)} survivors ===\n")

    for p in survivors:
        if not p.get("s2_id") and p.get("arxiv_id"):
            data = s2_get(f"/paper/ArXiv:{p['arxiv_id']}", {"fields": "paperId"})
            if data:
                p["s2_id"] = data.get("paperId")

    chase_papers = [p for p in survivors if p.get("s2_id")][:chase_top_n]
    phase3 = []

    for p in chase_papers:
        if len(all_papers) >= max_total:
            break
        print(f"  Chasing: {p['title'][:55]}")

        for ref in s2_references(p["s2_id"], limit=10):
            nr = normalize_s2(ref, f"ref_of|{p['title'][:20]}")
            if nr:
                nr["source_hypothesis"] = p.get("source_hypothesis", "")
                if dedup_add(nr, nr.get("source", "")):
                    phase3.append(nr)
                    print(f"    ← [{nr.get('year','')}] {nr['title'][:50]}")

        for cit in s2_citations(p["s2_id"], limit=10):
            nc = normalize_s2(cit, f"citer_of|{p['title'][:20]}")
            if nc:
                nc["source_hypothesis"] = p.get("source_hypothesis", "")
                if dedup_add(nc, nc.get("source", "")):
                    phase3.append(nc)
                    print(f"    → [{nc.get('year','')}] {nc['title'][:50]}")

    print(f"\n--- Phase 3: {len(phase3)} papers from citation chase ---")

    # === Phase 4: Critic filters Phase 3 papers ===
    if phase3:
        print(f"\n=== Phase 4: Critic Filtering Citation Chase ===\n")
        phase3_survivors = critic_filter(phase3, hypotheses, ps)
    else:
        phase3_survivors = []

    final = survivors + phase3_survivors
    print(f"\n{'='*60}")
    print(f"FINAL: {len(survivors)} from search + {len(phase3_survivors)} "
          f"from citations = {len(final)} papers")
    print(f"Dropped total: {len(phase1) + len(phase3) - len(final)} papers")
    print(f"{'='*60}")
    return final

# === RUN IT ===
papers = librarian_v2(hypotheses, ps=TEST_PS, depth=1, seeds_per_query=3, chase_top_n=5)

=== Phase 1: Keyword Search ===

[method] Can a retrieval-augmented generation approach be used to construc
  + [2025] AR-RAG: Autoregressive Retrieval Augmentation for Image Generatio
  + [2025] AI Agent-Driven Framework for Automated Product Knowledge Graph C
  + [2025] Intelligent Interaction Strategies for Context-Aware Cognitive Au
    S2 rate limited, waiting 5s...
  + [2022] Subsampling for Knowledge Graph Embedding Explained
  + [2021] Template-Based Graph Clustering
    S2 rate limited, waiting 5s...
  + [2024] TOBUGraph: Knowledge Graph-Based Retrieval for Enhanced LLM Perfo
  + [2026] RAG-Based Healthcare Query Assistant Using Graph Database: A Comp
  + [2025] Knowledge Graph Completion using RAG and Improved Structural Info
  + [2026] Tensor Manifold-Based Graph-Vector Fusion for AI-Native Academic 
  + [2024] Automated Literature Review Using NLP Techniques and LLM-Based Re
  + [2014] Unsupervised Visual and Textual Information Fusion in Multimedia 
    S2 rate limited, wa

In [48]:
# one-time cleanup
import os
for f in S2_CACHE.glob("*.json"):
    if f.read_text().strip() in ("", "null", "None"):
        f.unlink()
print("cleared stale S2 cache entries")

cleared stale S2 cache entries


In [53]:
import urllib.request

download_dir = STORE / "pdfs"
downloaded, skipped, no_url = 0, 0, 0

for p in papers:
    if not p.get("pdf_url"):
        no_url += 1
        continue
    
    # clean filename
    safe_name = "".join(c for c in p["title"][:60] if c.isalnum() or c in " -_") + ".pdf"
    safe_name = safe_name.strip().replace(" ", "_")
    dest = download_dir / safe_name
    
    if dest.exists():
        skipped += 1
        continue
    
    try:
        urllib.request.urlretrieve(p["pdf_url"], dest)
        downloaded += 1
        print(f"  ↓ {p['title'][:65]}")
        time.sleep(0.5)
    except Exception as e:
        print(f"  ✗ {p['title'][:45]}: {e}")

print(f"\n{downloaded} downloaded, {skipped} already existed, {no_url} had no PDF URL")
for p in papers:
    if not p.get("pdf_url") and p.get("s2_id"):
        data = s2_get(f"/paper/{p['s2_id']}", {"fields": "openAccessPdf"})
        if data and data.get("openAccessPdf"):
            p["pdf_url"] = data["openAccessPdf"]["url"]
            print(f"  Found: {p['title'][:50]} → {p['pdf_url'][:60]}")
        else:
            print(f"  No open access: {p['title'][:50]}")


0 downloaded, 19 already existed, 3 had no PDF URL
    S2 rate limited, waiting 5s...
  No open access: Knowledge Graph Completion using RAG and Improved 
    S2 rate limited, waiting 5s...
  No open access: The State of the Art of Natural Language Processin
    S2 rate limited, waiting 5s...
  No open access: Temporal Knowledge Graph Reasoning With Dynamic Me


In [56]:
reset_index()
ingest_folder(STORE / "pdfs", parse_cached, chunk_and_index)
print_manifest_summary()

index + manifest wiped


/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),
Token indices sequence length is longer than the specified maximum sequence length for this model (2309 > 512). Running this sequence through the model will result in indexing errors


  ingested: AI Agent-Driven Framework for Automated Product Knowledge Gr
  ingested: AI LITERATURE REVIEW SUITE
  ingested: A Multi-Agent System for Semantic Mapping of Relational Data
  ingested: <span id="page-13-1"></span>E Experimental Details for Extra
  ingested: Automated Literature Review Using NLP Techniques and LLM-Bas
  ingested: CoTKR: Chain-of-Thought Enhanced Knowledge Rewriting for Com
  ingested: Context Engineering for Multi-Agent LLM Code Assistants Usin
  ingested: From Local to Global: A GraphRAG Approach to Query-Focused S
  ingested: GAM-RAG: Gain-Adaptive Memory for Evolving Retrieval in Retr
  ingested: <span id="page-0-0"></span>Graph Chain-of-Thought: Augmentin


Recognizing Text: 100%|██████████| 34/34 [00:48<00:00,  1.41s/it]
/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: <span id="page-0-0"></span>Graph Retrieval-Augmented Generat


...

  ingested: TOBUGraph: Knowledge Graph-Based Retrieval for Enhanced LLM 


Recognizing tables: 100%|██████████| 1/1 [00:00<00:00,  3.76it/s]
Detecting bboxes: 0it [00:00, ?it/s]
/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: Template-Based Graph Clustering\*


Recognizing tables: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]
Detecting bboxes: 0it [00:00, ?it/s]
/tmp/ipykernel_58/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: Tensor Manifold-Based Graph-Vector Fusion for AI-Native Acad
  ingested: Corrective Retrieval Augmented Generation
  ingested: SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU

21 ingested, 0 skipped, 0 quarantined
embedder: BAAI/bge-base-en-v1.5  |  21 documents
  [ingested   ]   15 chunks  AI Agent-Driven Framework for Automated Product Knowled
  [ingested   ]   19 chunks  AI LITERATURE REVIEW SUITE
  [ingested   ]   11 chunks  A Multi-Agent System for Semantic Mapping of Relational
  [ingested   ]   69 chunks  <span id="page-13-1"></span>E Experimental Details for 
  [ingested   ]   24 chunks  Automated Literature Review Using NLP Techniques and LL
  [ingested   ]   70 chunks  CoTKR: Chain-of-Thought Enhanced Knowledge Rewriting fo
  [ingested   ]   27 chunks  Context Engineering for Multi-Agent LLM Code Assistants
  [ingested   ]   82 chunks  From Local to Global: A GraphRAG Approach to Query-Focu
  [ingested   ]   58 chunks  GAM-RAG: Gain-Adaptive Memory fo

In [57]:
data = collection.get(include=["documents"])
chunk_ids, docs = data["ids"], data["documents"]
doc_lookup = dict(zip(chunk_ids, docs))
bm25 = BM25Okapi([bm25_tok(d) for d in docs])
print(f"BM25 rebuilt: {len(chunk_ids)} chunks")

BM25 rebuilt: 1153 chunks


In [58]:
smoke("What is GraphRAG and how does it use community detection?")
smoke("How does corrective RAG handle low quality retrieval?")


Q: What is GraphRAG and how does it use community detection?
  → From Local to Global: A GraphRAG Approac | From Local to Global: A GraphRAG Approach to 
     GraphRAG contrasts with these approaches by generating a graph index from the source data, then applying graph-based community detection to create a thematic pa
  → From Local to Global: A GraphRAG Approac | From Local to Global: A GraphRAG Approach to 
     Given the graph index created in the previous step, a variety of community detection algorithms may be used to partition the graph into communities of strongly 
  → <span id="page-0-0"></span>Graph Retriev | 10.6 Standard Benchmarks
     GraphRAG is a relatively new field that lacks unified and standard benchmarks for evaluating different methods. Establishing a standard benchmark is crucial for

Q: How does corrective RAG handle low quality retrieval?
  → Corrective Retrieval Augmented Generatio | 2 Related Work
     Compared with recent studies [\(Schick et al.,](#page-11-

In [59]:
from typing import TypedDict, List, Dict, Literal, Optional
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
import json, re as _re

fast = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
big  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

class RAGState(TypedDict):
    question: str
    sub_queries: List[str]
    route: Literal["rag", "direct"]
    chunks: List[Dict]
    grade: Literal["sufficient", "insufficient", "irrelevant"]
    retries: int
    reformulated: List[str]
    answer: str
    citations: List[Dict]
    verified: bool
    trace: List[str]

# ---------- ROUTER ----------
ROUTE_PROMPT = """Classify this input:
- "rag": needs information from research papers to answer
- "direct": greeting, meta question, or general chat
Input: {question}
Return ONLY JSON: {{"route": "rag"}} or {{"route": "direct"}}"""

def route_node(state: RAGState) -> RAGState:
    out = fast.invoke(ROUTE_PROMPT.format(question=state["question"]))
    m = _re.search(r"\{.*\}", out.content, _re.DOTALL)
    state["route"] = json.loads(m.group())["route"] if m else "rag"
    state["trace"].append(f"route → {state['route']}")
    return state

# ---------- PLANNER ----------
PLAN_PROMPT = """Decompose this question into 1-3 standalone search queries.
Each query should retrieve relevant passages from research papers.
If the question is already specific enough, return it unchanged.
Question: {question}
Return ONLY JSON: {{"queries": ["...", ...]}}"""

def plan_node(state: RAGState) -> RAGState:
    out = fast.invoke(PLAN_PROMPT.format(question=state["question"]))
    m = _re.search(r"\{.*\}", out.content, _re.DOTALL)
    queries = json.loads(m.group())["queries"][:3] if m else [state["question"]]
    state["sub_queries"] = queries
    state["trace"].append(f"plan → {queries}")
    return state

# ---------- RETRIEVER ----------
def retrieve_node(state: RAGState) -> RAGState:
    all_chunks = {}
    k_each = 10 + (5 * state["retries"])   # widen on retries
    for q in state["sub_queries"]:
        cands = hybrid_retrieve(q, top_k=k_each)
        scores = reranker.predict([(q, doc_lookup[c]) for c in cands])
        for cid, score in zip(cands, scores):
            if cid not in all_chunks or score > all_chunks[cid]["rerank_score"]:
                meta = collection.get(ids=[cid], include=["metadatas"])["metadatas"][0]
                all_chunks[cid] = {
                    "id": cid, "text": doc_lookup[cid],
                    "rerank_score": float(score),
                    "title": meta.get("title", ""),
                    "section": meta.get("section", ""),
                }
    ranked = sorted(all_chunks.values(), key=lambda x: -x["rerank_score"])[:8]
    state["chunks"] = ranked
    state["trace"].append(f"retrieve → {len(ranked)} chunks (k_each={k_each})")
    return state

# ---------- GRADER (70B — precision critical) ----------
GRADE_PROMPT = """Question: {question}

Retrieved passages:
{chunks}

Judge the SET of passages as a whole. Passages being on-topic is NOT enough --
they must actually contain the information needed to answer THIS question.
If the passages discuss a DIFFERENT method/paper than the one asked about,
grade irrelevant even if the general topic overlaps.
- "sufficient": passages contain the information to answer fully
- "insufficient": on-topic but missing key parts
- "irrelevant": mostly off-topic
Return ONLY JSON: {{"grade": "...", "missing": "what is missing, if anything"}}"""

def grade_node(state: RAGState) -> RAGState:
    if not state["chunks"] or max(c["rerank_score"] for c in state["chunks"]) < -2:
        state["grade"] = "irrelevant"
        state["trace"].append("grade → irrelevant (rerank pre-filter)")
        return state
    chunks_text = "\n\n".join(
        f'[{i+1}] {c["text"][:600]}' for i, c in enumerate(state["chunks"]))
    out = big.invoke(GRADE_PROMPT.format(question=state["question"], chunks=chunks_text))
    m = _re.search(r"\{.*\}", out.content, _re.DOTALL)
    parsed = json.loads(m.group()) if m else {"grade": "insufficient", "missing": "parse fail"}
    state["grade"] = parsed["grade"]
    state["trace"].append(f"grade → {parsed['grade']} (missing: {parsed.get('missing','')})")
    return state

# ---------- REFORMULATOR ----------
REFORM_PROMPT = """The search failed to find sufficient passages.
Original question: {question}
Previous queries tried: {history}
What was missing: {missing}
Write ONE new search query using different vocabulary.
Return ONLY JSON: {{"query": "..."}}"""

def reformulate_node(state: RAGState) -> RAGState:
    missing = state["trace"][-1].split("missing: ")[-1].rstrip(")")
    out = fast.invoke(REFORM_PROMPT.format(
        question=state["question"],
        history=state["reformulated"],
        missing=missing))
    m = _re.search(r"\{.*\}", out.content, _re.DOTALL)
    new_q = json.loads(m.group())["query"] if m else state["question"]
    state["sub_queries"] = [new_q]
    state["reformulated"].append(new_q)
    state["retries"] += 1
    state["trace"].append(f"reformulate → '{new_q}' (retry {state['retries']})")
    return state

# ---------- GENERATOR (citation-enforced) ----------
GEN_PROMPT = """Answer using ONLY the passages below.
Rules:
- After every claim, cite the supporting passage like [2] or [1][3].
- If the passages do not contain the answer, say so explicitly.
- Do not use outside knowledge.

Passages:
{chunks}

Question: {question}"""

def generate_node(state: RAGState) -> RAGState:
    if state["grade"] in ("irrelevant",) and state["retries"] >= 3:
        state["answer"] = ("The indexed papers do not contain sufficient information "
                          "to answer this question. " + state["trace"][-1])
        state["citations"] = []
        state["trace"].append("generate → calibrated refusal")
        return state
    chunks_text = "\n\n".join(
        f'[{i+1}] ({c["title"][:40]} — {c["section"][:30]})\n{c["text"][:800]}'
        for i, c in enumerate(state["chunks"]))
    out = big.invoke(GEN_PROMPT.format(question=state["question"], chunks=chunks_text))
    state["answer"] = out.content
    state["citations"] = [{"index": i+1, "title": c["title"], "section": c["section"]}
                          for i, c in enumerate(state["chunks"])]
    state["trace"].append(f"generate → {len(state['answer'])} chars")
    return state

# ---------- DIRECT ANSWER (no retrieval needed) ----------
def direct_node(state: RAGState) -> RAGState:
    out = fast.invoke(state["question"])
    state["answer"] = out.content
    state["trace"].append("direct → answered without retrieval")
    return state

# ---------- WIRE THE GRAPH ----------
g = StateGraph(RAGState)
g.add_node("route", route_node)
g.add_node("plan", plan_node)
g.add_node("retrieve", retrieve_node)
g.add_node("grade", grade_node)
g.add_node("reformulate", reformulate_node)
g.add_node("generate", generate_node)
g.add_node("direct", direct_node)

g.set_entry_point("route")
g.add_conditional_edges("route", lambda s: s["route"],
    {"rag": "plan", "direct": "direct"})
g.add_edge("plan", "retrieve")
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", lambda s:
    "generate" if s["grade"] == "sufficient" or s["retries"] >= 3
    else "reformulate",
    {"generate": "generate", "reformulate": "reformulate"})
g.add_edge("reformulate", "retrieve")
g.add_edge("generate", END)
g.add_edge("direct", END)

agent = g.compile()
print("agent compiled ✓")

agent compiled ✓


In [60]:
def ask(question):
    result = agent.invoke({
        "question": question, "sub_queries": [], "route": "rag",
        "chunks": [], "grade": "sufficient", "retries": 0,
        "reformulated": [], "answer": "", "citations": [],
        "verified": False, "trace": []
    })
    print(f"\nQ: {question}")
    print(f"\nTRACE:")
    for t in result["trace"]:
        print(f"  {t}")
    print(f"\nANSWER:\n{result['answer'][:500]}")
    print(f"\nCITATIONS:")
    for c in result["citations"][:5]:
        print(f"  [{c['index']}] {c['title'][:50]} — {c['section'][:30]}")
    print("\n" + "="*70)
    return result

# test 1: answerable from corpus
ask("How does GraphRAG use community detection to summarize documents?")

# test 2: unanswerable — should trigger retries then refuse
ask("What does the RAPTOR paper propose for hierarchical tree summarization?")

# test 3: direct — should skip retrieval
ask("Hello, what can you help me with?")


Q: How does GraphRAG use community detection to summarize documents?

TRACE:
  route → rag
  plan → ['GraphRAG community detection algorithm', 'GraphRAG document summarization technique', 'GraphRAG community detection for text summarization']
  retrieve → 8 chunks (k_each=10)
  grade → sufficient (missing: None)
  generate → 632 chars

ANSWER:
GraphRAG uses community detection to summarize documents by first generating a graph index from the source data, then applying graph-based community detection to create a thematic partitioning of the data [3]. It partitions the graph into a hierarchy of communities of closely related entities [8]. Next, it uses an LLM to generate community-level summaries in a bottom-up manner following the hierarchical structure of extracted communities, with summaries at higher levels of the hierarchy recursiv

CITATIONS:
  [1] From Local to Global: A GraphRAG Approach to Query — 4 Analysis > 7 Conclusion
  [2] Retrieval-Augmented Generation for Natural Langua

{'question': 'Hello, what can you help me with?',
 'sub_queries': [],
 'route': 'direct',
 'chunks': [],
 'grade': 'sufficient',
 'retries': 0,
 'reformulated': [],
 'answer': "I can be used in a variety of ways, from answering questions and providing information on a particular topic, to generating text and even creating art. I'm here to help with any questions or tasks you may have. What's on your mind?",
 'citations': [],
 'verified': False,
 'trace': ['route → direct', 'direct → answered without retrieval']}

In [62]:
r = ask("What is GraphRAG?")


Q: What is GraphRAG?

TRACE:
  route → rag
  plan → ['GraphRAG definition', 'GraphRAG research papers', 'GraphRAG architecture']
  retrieve → 8 chunks (k_each=10)
  grade → sufficient (missing: none)
  generate → 436 chars

ANSWER:
GraphRAG is a framework that leverages external structured knowledge graphs to improve contextual understanding of LMs and generate more informed responses [1]. It is a relatively new field that lacks unified and standard benchmarks for evaluating different methods [2]. GraphRAG methods leverage explicit entity and relationship representations in graph data, enabling precise answers by retrieving relevant structured information [3].

CITATIONS:
  [1] <span id="page-0-0"></span>Graph Retrieval-Augment — 3 Preliminaries > 3.2 Graph Ne
  [2] <span id="page-0-0"></span>Graph Retrieval-Augment — 10.6 Standard Benchmarks
  [3] <span id="page-0-0"></span>Graph Retrieval-Augment — 1 Introduction
  [4] <span id="page-0-0"></span>Graph Retrieval-Augment — 9.1 Downstr

In [63]:
r = verify_answer(r)
print("VERIFICATION:", r["trace"][-1])

VERIFICATION: verify → all claims supported ✓


In [64]:
def synthesize(ps, hypotheses):
    print(f"Synthesizing report for {len(hypotheses)} hypotheses...\n")
    sections = []
    
    for i, h in enumerate(hypotheses):
        print(f"\n{'='*60}")
        print(f"H{i+1} [{h['section_type']}]: {h['hypothesis'][:70]}")
        print(f"{'='*60}")
        
        result = agent.invoke({
            "question": h["hypothesis"],
            "sub_queries": h["search_queries"],
            "route": "rag",
            "chunks": [], "grade": "sufficient", "retries": 0,
            "reformulated": [], "answer": "", "citations": [],
            "verified": False, "trace": []
        })
        
        # verify
        result = verify_answer(result)
        
        sections.append({
            "hypothesis": h["hypothesis"],
            "section_type": h["section_type"],
            "answer": result["answer"],
            "citations": result["citations"],
            "verified": result["verified"],
            "trace": result["trace"],
            "grade": result["grade"],
        })
        
        status = "✓ verified" if result["verified"] else "⚠ unverified"
        print(f"\n  [{status}] {len(result['answer'])} chars, "
              f"{len(result['citations'])} citations, "
              f"{len(result['trace'])} trace steps")
    
    # --- Assemble the report ---
    report = f"# Research Synthesis Report\n\n**Problem Statement:** {ps}\n\n"
    
    for i, s in enumerate(sections):
        report += f"## {i+1}. {s['hypothesis']}\n\n"
        if s["grade"] in ("irrelevant",) and not s["answer"].strip():
            report += "*Not covered by the discovered literature.*\n\n"
        else:
            report += s["answer"] + "\n\n"
            if s["citations"]:
                report += "**Sources:**\n"
                for c in s["citations"][:5]:
                    report += f"- [{c['index']}] {c['title'][:60]} — {c['section'][:40]}\n"
                report += "\n"
    
    # --- Honest gaps appendix ---
    gaps = [s for s in sections if s["grade"] in ("irrelevant", "insufficient") 
            and s.get("retries", 0) >= 3]
    if gaps:
        report += "## Gaps in Coverage\n\n"
        report += "The following hypotheses could not be fully addressed:\n\n"
        for s in gaps:
            report += f"- {s['hypothesis'][:80]}\n"
        report += "\n"
    
    print(f"\n{'='*60}")
    print(f"REPORT: {len(sections)} sections, "
          f"{sum(len(s['citations']) for s in sections)} total citations, "
          f"{sum(s['verified'] for s in sections)}/{len(sections)} verified")
    print(f"{'='*60}")
    
    return report, sections

# --- RUN IT ---
report, sections = synthesize(TEST_PS, hypotheses)
print("\n\n" + report[:3000])

Synthesizing report for 5 hypotheses...


H1 [method]: Can a retrieval-augmented generation approach be used to construct a k

  [⚠ unverified] 489 chars, 8 citations, 9 trace steps

H2 [sota]: How can a cross-encoder based reranking approach be used to efficientl

  [⚠ unverified] 836 chars, 8 citations, 6 trace steps

H3 [feasibility]: Is it feasible to use a multi-agent framework with large language mode

  [⚠ unverified] 92 chars, 8 citations, 6 trace steps

H4 [method]: Can a graph neural network-based approach be used to construct a knowl

  [⚠ unverified] 609 chars, 8 citations, 15 trace steps

H5 [sota]: What are the current state-of-the-art approaches for automated researc

  [⚠ unverified] 81 chars, 8 citations, 15 trace steps

REPORT: 5 sections, 40 total citations, 0/5 verified


# Research Synthesis Report

**Problem Statement:** Build an agentic framework for automated research synthesis in technical
competitions. The system should browse research papers on the web based 